In [27]:
# =====================================================
# 05 Model Training with MLflow - Rider-Level Customer Churn
# =====================================================

import os
import time
import json
import warnings
import joblib

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

sns.set_theme(style="whitegrid")

# =====================================================
# Create Folders
# =====================================================

os.makedirs("../models", exist_ok=True)
os.makedirs("mlflow_artifacts", exist_ok=True)

# =====================================================
# MLflow Experiment Setup
# =====================================================

mlflow.set_experiment("RideWise_Customer_Churn_Training")

# =====================================================
# Load Rider-Level Feature Dataset
# =====================================================

df = pd.read_csv("../data/processed/rider_level_churn_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())

# =====================================================
# Define Target
# =====================================================

target = "is_churned"
y = df[target]

# =====================================================
# Drop Columns Not Needed for Modelling
# =====================================================

drop_cols = [
    "is_churned",
    "user_id",
    "signup_date",
    "referred_by"
]

X = df.drop(columns=drop_cols, errors="ignore")

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts(normalize=True) * 100)

assert all(col not in X.columns for col in drop_cols), "Some drop columns are still in X."
print("\nValidation passed: unwanted columns removed.")

# =====================================================
# Identify Numerical and Categorical Features
# =====================================================

categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

# =====================================================
# Train-Test Split - Keep 80/20 only
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# =====================================================
# Preprocessing Pipeline
# =====================================================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# =====================================================
# Evaluation Function
# =====================================================

def evaluate_model(model_name, model, X_test, y_test, threshold=0.50):
    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)

    return {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1 Score": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs)
    }

# =====================================================
# Artifact Logging Function
# =====================================================

def log_artifacts(model_name, model, X_test, y_test, threshold):
    safe_name = model_name.lower().replace(" ", "_")

    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)

    report = classification_report(y_test, preds, zero_division=0)

    report_path = f"mlflow_artifacts/{safe_name}_classification_report.txt"
    with open(report_path, "w") as f:
        f.write(report)
    mlflow.log_artifact(report_path, artifact_path="reports")

    cm_path = f"mlflow_artifacts/{safe_name}_confusion_matrix.png"
    ConfusionMatrixDisplay.from_predictions(y_test, preds)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.savefig(cm_path, bbox_inches="tight")
    mlflow.log_artifact(cm_path, artifact_path="plots")
    plt.close()

    roc_path = f"mlflow_artifacts/{safe_name}_roc_curve.png"
    RocCurveDisplay.from_predictions(y_test, probs)
    plt.title(f"ROC Curve - {model_name}")
    plt.savefig(roc_path, bbox_inches="tight")
    mlflow.log_artifact(roc_path, artifact_path="plots")
    plt.close()

    pr_path = f"mlflow_artifacts/{safe_name}_precision_recall_curve.png"
    PrecisionRecallDisplay.from_predictions(y_test, probs)
    plt.title(f"Precision-Recall Curve - {model_name}")
    plt.savefig(pr_path, bbox_inches="tight")
    mlflow.log_artifact(pr_path, artifact_path="plots")
    plt.close()

# =====================================================
# Build Models
# =====================================================

scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=42
            ))
        ]
    ),

    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=5,
                min_samples_split=10,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            ))
        ]
    ),

    "XGBoost": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", XGBClassifier(
                n_estimators=500,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.9,
                colsample_bytree=0.9,
                min_child_weight=5,
                gamma=1,
                scale_pos_weight=scale_pos_weight,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            ))
        ]
    )
}

# =====================================================
# Cross-Validation, Training, Threshold Tuning and MLflow Logging
# =====================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

thresholds = np.arange(0.20, 0.61, 0.05)

cv_results = []
model_results = []
threshold_results = []

best_model = None
best_model_name = None
best_threshold = None
best_f1 = 0

for name, model in models.items():

    start_time = time.time()

    with mlflow.start_run(run_name=name):

        # -------------------------
        # Cross-validation
        # -------------------------

        cv_scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=cv,
            scoring="roc_auc",
            n_jobs=-1
        )

        mean_cv_auc = cv_scores.mean()
        std_cv_auc = cv_scores.std()

        cv_results.append({
            "Model": name,
            "Mean ROC-AUC": mean_cv_auc,
            "Std ROC-AUC": std_cv_auc
        })

        # -------------------------
        # Train model
        # -------------------------

        model.fit(X_train, y_train)

        # -------------------------
        # Evaluate at default threshold 0.50
        # -------------------------

        default_result = evaluate_model(
            name,
            model,
            X_test,
            y_test,
            threshold=0.50
        )

        model_results.append(default_result)

        # -------------------------
        # Threshold tuning
        # -------------------------

        probs = model.predict_proba(X_test)[:, 1]

        model_threshold_results = []

        for threshold in thresholds:
            preds = (probs >= threshold).astype(int)

            result = {
                "Model": name,
                "Threshold": threshold,
                "Accuracy": accuracy_score(y_test, preds),
                "Precision": precision_score(y_test, preds, zero_division=0),
                "Recall": recall_score(y_test, preds, zero_division=0),
                "F1 Score": f1_score(y_test, preds, zero_division=0),
                "ROC-AUC": roc_auc_score(y_test, probs)
            }

            threshold_results.append(result)
            model_threshold_results.append(result)

        model_best_row = (
            pd.DataFrame(model_threshold_results)
            .sort_values(by="F1 Score", ascending=False)
            .iloc[0]
        )

        model_best_threshold = model_best_row["Threshold"]
        model_best_f1 = model_best_row["F1 Score"]

        training_time = time.time() - start_time

        # -------------------------
        # Log MLflow tags
        # -------------------------

        mlflow.set_tags({
            "project": "RideWise Customer Churn Prediction",
            "notebook": "05_model_training",
            "tracking_level": "Level 2 MLflow",
            "model_type": name,
            "target": target,
            "split": "80/20",
            "author": "Batholomew Ohanme"
        })

        # -------------------------
        # Log parameters
        # -------------------------

        mlflow.log_param("model_name", name)
        mlflow.log_param("dataset", "../data/processed/rider_level_churn_dataset.csv")
        mlflow.log_param("target_column", target)
        mlflow.log_param("test_size", 0.2)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("threshold_range", "0.20_to_0.60")
        mlflow.log_param("best_threshold", model_best_threshold)
        mlflow.log_param("train_rows", X_train.shape[0])
        mlflow.log_param("test_rows", X_test.shape[0])
        mlflow.log_param("number_of_features", X_train.shape[1])

        # -------------------------
        # Log metrics
        # -------------------------

        mlflow.log_metric("cv_mean_roc_auc", mean_cv_auc)
        mlflow.log_metric("cv_std_roc_auc", std_cv_auc)

        mlflow.log_metric("accuracy_threshold_0_50", default_result["Accuracy"])
        mlflow.log_metric("precision_threshold_0_50", default_result["Precision"])
        mlflow.log_metric("recall_threshold_0_50", default_result["Recall"])
        mlflow.log_metric("f1_threshold_0_50", default_result["F1 Score"])
        mlflow.log_metric("roc_auc", default_result["ROC-AUC"])

        mlflow.log_metric("best_threshold_f1", model_best_f1)
        mlflow.log_metric("training_time_seconds", training_time)

        # -------------------------
        # Log reports and plots using best threshold
        # -------------------------

        log_artifacts(
            model_name=name,
            model=model,
            X_test=X_test,
            y_test=y_test,
            threshold=model_best_threshold
        )

        # -------------------------
        # Log model to MLflow
        # -------------------------

        input_example = X_train.head(5)

        signature = infer_signature(
            X_train,
            model.predict(X_train)
        )

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=input_example,
            skops_trusted_types=[
                "xgboost.core.Booster",
                "xgboost.sklearn.XGBClassifier"
            ]
        )

        print(f"\n{name}")
        print("-" * 40)
        print(f"CV ROC-AUC: {mean_cv_auc:.4f} ± {std_cv_auc:.4f}")
        print(f"Default F1 @ 0.50: {default_result['F1 Score']:.4f}")
        print(f"Best Threshold: {model_best_threshold}")
        print(f"Best F1: {model_best_f1:.4f}")

        # -------------------------
        # Track overall best model
        # -------------------------

        if model_best_f1 > best_f1:
            best_f1 = model_best_f1
            best_model = model
            best_model_name = name
            best_threshold = model_best_threshold

# =====================================================
# Display Results
# =====================================================

cv_results_df = pd.DataFrame(cv_results)
model_results_df = pd.DataFrame(model_results)
threshold_results_df = pd.DataFrame(threshold_results)

print("\nCross-Validation ROC-AUC Results:")
display(cv_results_df.sort_values(by="Mean ROC-AUC", ascending=False))

print("\nModel Comparison at Threshold 0.50:")
display(model_results_df.sort_values(by="ROC-AUC", ascending=False))

print("\nTop Threshold Tuning Results:")
display(threshold_results_df.sort_values(by="F1 Score", ascending=False).head(15))

print("\nBest Model Selection")
print("-" * 40)
print("Best model:", best_model_name)
print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

# =====================================================
# Save Best Model and Threshold
# =====================================================

joblib.dump(best_model, "../models/churn_prediction_model.pkl")
joblib.dump(best_threshold, "../models/churn_prediction_threshold.pkl")

print("\nBest model and threshold saved successfully.")
print(f"Saved model: {best_model_name}")
print(f"Saved threshold: {best_threshold}")

Dataset shape: (10000, 48)


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,referred_by,is_referred,total_trips,avg_fare,...,tip_ratio,spend_per_day,session_intensity,app_usage_score,conversion_engagement_score,non_standard_trip_rate,surge_fare_interaction,loyalty_spend_interaction,age_group,is_churned
0,R00000,2025-01-24 00:00:00+00:00,Bronze,34.729629,Nairobi,5.0,R00001,1,25,14.642000,...,0.011009,3.936022,0.043011,276.000000,9.75,0.520000,16.047632,98.400538,26-35,0
1,R00001,2024-09-09 00:00:00+00:00,Bronze,34.571020,Nairobi,4.7,Unknown,0,14,12.895000,...,0.004210,0.784913,0.013043,465.777778,0.00,0.571429,13.816071,10.988783,26-35,0
2,R00002,2024-09-07 00:00:00+00:00,Bronze,47.133960,Lagos,4.2,Unknown,0,24,15.791250,...,0.013747,1.633578,0.012931,573.000000,0.00,0.291667,18.817906,39.205862,46-60,0
3,R00003,2025-03-17 00:00:00+00:00,Bronze,41.658628,Nairobi,4.9,Unknown,0,9,13.496667,...,0.007162,2.962683,0.073171,125.555556,0.00,1.000000,15.596148,26.664146,36-45,1
4,R00004,2024-08-20 00:00:00+00:00,Silver,40.681709,Lagos,3.9,R00002,1,16,16.776875,...,0.034944,1.073720,0.008000,42.500000,0.00,0.687500,21.180805,17.179520,36-45,0


Feature shape: (10000, 44)
Target shape: (10000,)

Target distribution:
is_churned
0    81.14
1    18.86
Name: proportion, dtype: float64

Validation passed: unwanted columns removed.

Categorical features:
['loyalty_status', 'city', 'favourite_payment_type', 'favourite_weather', 'favourite_vehicle_type', 'age_group']

Numerical features:
['age', 'avg_rating_given', 'is_referred', 'total_trips', 'avg_fare', 'total_fare', 'avg_surge', 'avg_tip', 'avg_tip_percentage', 'avg_trip_duration', 'avg_fare_per_minute', 'avg_driver_rating', 'avg_driver_acceptance', 'driver_info_missing_rate', 'weekend_trip_rate', 'peak_hour_trip_rate', 'night_trip_rate', 'surge_trip_rate', 'total_sessions', 'avg_time_on_app', 'avg_pages_visited', 'conversion_rate', 'account_tenure_days', 'trip_velocity', 'trips_per_session', 'engagement_score', 'avg_days_between_trips', 'fare_per_session', 'trips_per_active_day', 'driver_quality_score', 'tip_ratio', 'spend_per_day', 'session_intensity', 'app_usage_score', 'conver


Logistic Regression
----------------------------------------
CV ROC-AUC: 0.6091 ± 0.0116
Default F1 @ 0.50: 0.3488
Best Threshold: 0.49999999999999994
Best F1: 0.3488



Random Forest
----------------------------------------
CV ROC-AUC: 0.6054 ± 0.0060
Default F1 @ 0.50: 0.2913
Best Threshold: 0.39999999999999997
Best F1: 0.3538


...


Model Comparison at Threshold 0.50:


,Model,Threshold,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.5,0.5780,0.245919,0.599469,0.348765,0.620042
1,Random Forest,0.5,0.7275,0.285714,0.297082,0.291287,0.615492
2,XGBoost,0.5,0.6175,0.244737,0.493369,0.327177,0.608550



Top Threshold Tuning Results:


,Model,Threshold,Accuracy,Precision,Recall,F1 Score,ROC-AUC
23,XGBoost,0.45,0.5490,0.246377,0.676393,0.361190,0.608550
13,Random Forest,0.40,0.5270,0.238270,0.687003,0.353825,0.615492
6,Logistic Regression,0.50,0.5780,0.245919,0.599469,0.348765,0.620042
5,Logistic Regression,0.45,0.4745,0.226016,0.737401,0.345986,0.620042
4,Logistic Regression,0.40,0.3895,0.215250,0.846154,0.343195,0.620042
22,XGBoost,0.40,0.4380,0.217260,0.761273,0.338045,0.608550
11,Random Forest,0.30,0.3345,0.207721,0.899204,0.337481,0.615492
21,XGBoost,0.35,0.3595,0.209512,0.864721,0.337300,0.608550
3,Logistic Regression,0.35,0.3135,0.206021,0.925729,0.337035,0.620042
12,Random Forest,0.35,0.4025,0.212377,0.801061,0.335742,0.615492



Best Model Selection
----------------------------------------
Best model: XGBoost
Best threshold: 0.44999999999999996
Best F1: 0.3611898016997167

Best model and threshold saved successfully.
Saved model: XGBoost
Saved threshold: 0.44999999999999996
